### "Mas como sabemos o modelo se ainda nem exploramos os dados da SUSEP?"

Esse notebook serve para responder essa pergunta e nos direcionar para um bom entendimento da base e permitir que criemos um excelente modelo


In [0]:
landing_path = "/Volumes/eb_02_brazilian_insurance/00_landing/susep_raw/"

display(
    dbutils.fs.ls(landing_path)
)

path,name,size,modificationTime
dbfs:/Volumes/eb_02_brazilian_insurance/00_landing/susep_raw/Ses_cias.csv,Ses_cias.csv,41013,1795209286000
dbfs:/Volumes/eb_02_brazilian_insurance/00_landing/susep_raw/Ses_ramos.csv,Ses_ramos.csv,7126,1795209286000
dbfs:/Volumes/eb_02_brazilian_insurance/00_landing/susep_raw/Ses_seguros.csv,Ses_seguros.csv,137419201,1795209302000


In [0]:
Expldf_cias = (
    spark.read
        .option("header", True)
        .option("sep", ";")
        .csv(landing_path + "Ses_cias.csv")
)

In [0]:
df_ramos = (
    spark.read
        .option("header", True)
        .option("sep", ";")
        .csv(landing_path + "Ses_ramos.csv")
)

In [0]:
df_seguros = (
    spark.read
        .option("header", True)
        .option("sep", ";")
        .csv(landing_path + "Ses_seguros.csv")
)
 

In [0]:
display(df_cias)


Coenti,Noenti,Cogrupo,Nogrupo
03727,FAIRWAY SEGUROS S.A.,,null
03786,ARCA SEGURADORA S.A,,null
03794,BS2 SEGUROS S.A.,,null
03816,REAL SEGUROS VIDA E PREVID�NCIA S.A. (ATUAL DENOMINA��O DE REAL TOKIO MARINE VIDA E PREVID�NCIA S.A.),,null
03956,FAM SEGURADORA DE CR�DITO E GARANTIA S.A.,,null
03964,LIBERTY MUTUAL SURETY BRASIL SEGUROS S.A.,,null
04031,XP VIDA E PREVID�NCIA S.A.,,null
04111,JNS SEGURADORA S.A.,,null
04120,88I SEGURADORA DIGITAL S.A.,,null
04189,PO SEGURADORA S.A.,,null


In [0]:
display(df_ramos)


In [0]:
display(df_seguros)

### Estrutura dos dados

In [0]:
df_cias.printSchema()

df_ramos.printSchema()

df_seguros.printSchema()

root
 |-- Coenti: string (nullable = true)
 |-- Noenti: string (nullable = true)
 |-- Cogrupo: string (nullable = true)
 |-- Nogrupo: string (nullable = true)

root
 |-- coramo: string (nullable = true)
 |-- noramo: string (nullable = true)

root
 |-- damesano: string (nullable = true)
 |-- coenti: string (nullable = true)
 |-- cogrupo: string (nullable = true)
 |-- coramo: string (nullable = true)
 |-- premio_direto: string (nullable = true)
 |-- premio_de_seguros: string (nullable = true)
 |-- premio_retido: string (nullable = true)
 |-- premio_ganho: string (nullable = true)
 |-- sinistro_direto: string (nullable = true)
 |-- sinistro_retido: string (nullable = true)
 |-- desp_com: string (nullable = true)
 |-- premio_emitido2: string (nullable = true)
 |-- premio_emitido_cap: string (nullable = true)
 |-- despesa_resseguros: string (nullable = true)
 |-- sinistro_ocorrido: string (nullable = true)
 |-- receita_resseguro: string (nullable = true)
 |-- sinistros_ocorridos_cap: string

### Volume dos dados

In [0]:
print("cias:", df_cias.count())

print("ramos:", df_ramos.count())

print("seguros:", df_seguros.count())

cias: 770
ramos: 161
seguros: 1818607


### Lista de colunas do fato

In [0]:
for col in df_seguros.columns:
    print(col)

### Análise da Fato

In [0]:
from pyspark.sql.functions import col

duplicados = (
    df_seguros
        .groupBy(
            "damesano",
            "coenti",
            "coramo"
        )
        .count()
        .filter(col("count") > 1)
)

display(duplicados)

###  Chave da seguradora

In [0]:
from pyspark.sql.functions import count

display(
    df_cias
      .groupBy("coenti")
      .count()
      .filter("count > 1")
)

### Chave do ramo

In [0]:
display(
    df_ramos
      .groupBy("coramo")
      .count()
      .filter("count > 1")
)